In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
ls -l /content/drive/MyDrive/ColabNotebooks/ACIIDS2023GIFT4Rec/

total 5711
drwx------ 3 root root    4096 Dec 11 10:16 data_loader/
-rw------- 1 root root 1506636 May  9  2024 do_an_final.pdf
-rw------- 1 root root  101545 May  9  2024 douban_eda_one_task_do_an.ipynb
-rw------- 1 root root 1018189 May  9  2024 douban_training_model.ipynb
-rw------- 1 root root     737 Dec  8 13:29 huong_dan_chay.txt
-rw------- 1 root root  586918 May  9  2024 load_melu.ipynb
-rw------- 1 root root   26961 May  9  2024 main_gift4rec_advanced_kgat.py
-rw------- 1 root root   32355 May  9  2024 main_gift4rec_simple_kgat.py
-rw------- 1 root root   15150 May  9  2024 main_kgat_2_dropoutnet.py
-rw------- 1 root root   20015 May  9  2024 main_kgat_2_raw_new_ver.py
-rw------- 1 root root    7235 May  9  2024 mask_optimization_gift4u.py
-rw------- 1 root root  156988 Dec 24 03:34 ml_1m_eda_one_task_do_an.ipynb
-rw------- 1 root root 2355241 May  9  2024 ml_1m_training_model.ipynb
drwx------ 3 root root    4096 Dec 11 10:16 model/
drwx------ 2 root root    4096 Dec 11 10:16

In [ ]:
%cd /content/drive/MyDrive/ColabNotebooks/

/content/drive/MyDrive/ColabNotebooks


In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
import os

In [120]:
df_train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/ACIIDS2023GIFT4Rec/ml-1m/ratings.dat', sep = '::', names = ["user id", "item id", "rating", "timestamp"])

/tmp/ipython-input-225787063.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_train = pd.read_csv('/content/drive/MyDrive/ColabNotebooks/ACIIDS2023GIFT4Rec/ml-1m/ratings.dat', sep = '::', names = ["user id", "item id", "rating", "timestamp"])


In [ ]:
!wget http://files.grouplens.org/datasets/movielens/ml-1m.zip
!unzip ml-1m.zip
df_train = pd.read_csv('ml-1m/ratings.dat', sep='::', names=["user_id", "item_id", "rating", "timestamp"], engine='python')

--2025-12-24 03:40:37--  http://files.grouplens.org/datasets/movielens/ml-1m.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://files.grouplens.org/datasets/movielens/ml-1m.zip [following]
--2025-12-24 03:40:37--  https://files.grouplens.org/datasets/movielens/ml-1m.zip
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5917549 (5.6M) [application/zip]
Saving to: ‘ml-1m.zip’

ml-1m.zip           100%[===================>]   5.64M  15.5MB/s    in 0.4s    

2025-12-24 03:40:38 (15.5 MB/s) - ‘ml-1m.zip’ saved [5917549/5917549]

Archive:  ml-1m.zip
   creating: ml-1m/
  inflating: ml-1m/movies.dat        
  inflating: ml-1m/ratings.dat       
  inflating: ml-1m/README            
  inflating: ml-1m/users.dat 

In [ ]:
df_user = pd.read_csv(
    'ml-1m/users.dat'
, sep = '::'
, names = ['user', 'gender', 'age', 'occupation', 'zip_code']
, encoding='latin-1'
, engine='python' )

In [ ]:
df_train = df_train[df_train['rating'] >= 3]

In [ ]:
print(df_train.columns.tolist())

['user_id', 'item_id', 'rating', 'timestamp']


In [ ]:
df_train['user_id'].unique()

array([   1,    2,    3, ..., 6038, 6039, 6040])

In [ ]:
df_train['item_id'].max()

3952

In [ ]:
df_train['user'] = df_train['user id'].copy()
df_train['item'] = df_train['item id'].copy()

<ipython-input-99-42ede7b853c7>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['user'] = df_train['user id'].copy()
<ipython-input-99-42ede7b853c7>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['item'] = df_train['item id'].copy()


In [ ]:
df_train.loc[:, 'user'] = df_train['user_id']
df_train.loc[:, 'item'] = df_train['item_id']

In [ ]:
path_save = '/content/drive/MyDrive/ColabNotebooks/ACIIDS2023GIFT4Rec/datasets/ml_1m_one_task'

In [ ]:
df_train_groupby_user = df_train.groupby('user', as_index = False).agg(lambda x : len(x))

In [ ]:
#split = 0.95

In [ ]:
split = 0.8

In [ ]:
df_train_groupby_user['n_interactions'] = df_train_groupby_user['item'].copy()

In [ ]:
list_users = df_train_groupby_user['user'][np.argsort(-df_train_groupby_user['item'].values)][:int(split * len(df_train_groupby_user))]

In [ ]:
list_users

,user
4167,4169
4275,4277
1679,1680
3616,3618
1014,1015
...,...
2048,2049
2074,2075
5871,5873
1866,1867


In [ ]:
list_users_cold_start = df_train_groupby_user['user'][np.argsort(-df_train_groupby_user['item'].values)][int(split * len(df_train_groupby_user)):]

In [ ]:
dict_user_to_id = {value : key + 1 for key, value in enumerate(list_users)}

In [ ]:
max_user_id = max(list(dict_user_to_id.values()))

In [ ]:
dict_user_to_id.update({value : key + 1 + max_user_id for key, value in enumerate(list_users_cold_start)})

In [ ]:
list_users = np.arange(list_users.shape[0]) + 1

In [ ]:
list_users_cold_start = np.arange(list_users_cold_start.shape[0]) + 1 + max_user_id

In [ ]:
list_users_missing = [user for user in df_user['user'].unique() if user not in dict_user_to_id]
max_user_id = max(list(dict_user_to_id.values()))
dict_user_to_id.update({value : key + 1 + max_user_id for key, value in enumerate(list_users_missing)})

In [ ]:
df_train

,user_id,item_id,rating,timestamp,user,item
0,1,1193,5,978300760,1,1193
1,1,661,3,978302109,1,661
2,1,914,3,978301968,1,914
3,1,3408,4,978300275,1,3408
4,1,2355,5,978824291,1,2355
...,...,...,...,...,...,...
1000203,6040,1090,3,956715518,6040,1090
1000205,6040,1094,5,956704887,6040,1094
1000206,6040,562,5,956704746,6040,562
1000207,6040,1096,4,956715648,6040,1096


In [ ]:
df_train['user'] = df_train['user'].map(dict_user_to_id)

In [ ]:
df_train['user_item'] = df_train['user'].astype(str) + "|" + df_train['item'].astype(str)

In [ ]:
df_train

,user_id,item_id,rating,timestamp,user,item,user_item
0,1,1193,5,978300760,3876,1193,3876|1193
1,1,661,3,978302109,3876,661,3876|661
2,1,914,3,978301968,3876,914,3876|914
3,1,3408,4,978300275,3876,3408,3876|3408
4,1,2355,5,978824291,3876,2355,3876|2355
...,...,...,...,...,...,...,...
1000203,6040,1090,3,956715518,842,1090,842|1090
1000205,6040,1094,5,956704887,842,1094,842|1094
1000206,6040,562,5,956704746,842,562,842|562
1000207,6040,1096,4,956715648,842,1096,842|1096


In [ ]:
split_cold_warm = 0.5
len_new_user = len(list_users_cold_start)

In [ ]:
df_train_groupby_user['n_interactions'] = df_train_groupby_user['item'].copy()
#old
#df_train_groupby_user['n_interactions'][df_train_groupby_user['user'].isin(list_users) == False] = 0
df_train_groupby_user['n_interactions'][(df_train_groupby_user['user'].isin(list_users) == False) & (df_train_groupby_user['user'].isin(list_users_cold_start[:int(split_cold_warm * len_new_user)]) == False)] = 0

/tmp/ipython-input-3859622317.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_train_groupby_user['n_interactions'][(df_train_groupby_user['user'].isin(list_users) == False) & (df_train_groupby_user['user'].isin(list_users_cold_start[:

In [ ]:
df_test_cold = df_train[df_train['user'].isin(list_users_cold_start[int(split_cold_warm * len_new_user):])]

In [ ]:
df_test_cold

,user_id,item_id,rating,timestamp,user,item,user_item
233,4,3468,5,978294008,5756,3468,5756|3468
234,4,1210,3,978293924,5756,1210,5756|1210
235,4,2951,4,978294282,5756,2951,5756|2951
236,4,1214,4,978294260,5756,1214,5756|1214
237,4,1036,4,978294282,5756,1036,5756|1036
...,...,...,...,...,...,...,...
999738,6038,1296,5,956714684,5892,1296,5892|1296
999739,6038,1354,3,956714725,5892,1354,5892|1354
999742,6038,2716,3,956707604,5892,2716,5892|2716
999743,6038,3396,3,956706827,5892,3396,5892|3396


In [ ]:
df_test_warm = df_train[df_train['user'].isin(list_users_cold_start[:int(split_cold_warm * len_new_user)])]

In [ ]:
df_test_warm

,user_id,item_id,rating,timestamp,user,item,user_item
523,7,648,4,978234737,4943,648,4943|648
524,7,861,4,978234874,4943,861,4943|861
525,7,2916,5,978234842,4943,2916,4943|2916
526,7,3578,3,978234737,4943,3578,4943|3578
527,7,3793,3,978234737,4943,3793,4943|3793
...,...,...,...,...,...,...,...
997999,6029,2959,4,956722071,5052,2959,5052|2959
998000,6029,2628,3,956722153,5052,2628,5052|2628
998001,6029,2687,4,956722090,5052,2687,5052|2687
998002,6029,899,5,956721639,5052,899,5052|899


In [ ]:
df_train_copy = df_train[df_train['user'].isin(list_users) | df_train['user'].isin(list_users_cold_start[:int(split_cold_warm * len_new_user)])]

In [ ]:
df_test_warm = df_test_warm.sort_values(by = ['timestamp']).groupby('user', as_index = False).agg(lambda x : [elem for elem in x][-1])

In [ ]:
df_train_copy = df_train_copy[df_train_copy['user_item'].isin(df_test_warm['user_item'].unique()) == False]

In [ ]:
df_train = df_train[df_train['user'].isin(list_users) | df_train['user'].isin(list_users_cold_start[:int(split_cold_warm * len_new_user)])]

In [ ]:
#split_val = 0.9

In [ ]:
#df_val = df_train.sort_values(by = ['timestamp']).groupby('user', as_index = False).agg(lambda x : [elem for elem in x][int(split_val * len([elem for elem in x])):])

In [ ]:
#df_val = df_val.set_index(['user']).apply(pd.Series.explode).reset_index()

In [ ]:
#df_train = df_train.sort_values(by = ['timestamp']).groupby('user', as_index = False).agg(lambda x : [elem for elem in x][:int(split_val * len([elem for elem in x]))])

In [ ]:
#df_train = df_train.set_index(['user']).apply(pd.Series.explode).reset_index()

In [ ]:
#df_train = df_train[df_train['user_item'].isin(df_val['user_item'].unique()) == False]

In [ ]:
df_train_groupby_user['n_interactions'] = df_train_groupby_user['item'].copy()
#old
#df_train_groupby_user['n_interactions'][df_train_groupby_user['user'].isin(list_users) == False] = 0
df_train_groupby_user['n_interactions'][(df_train_groupby_user['user'].isin(list_users) == False) & (df_train_groupby_user['user'].isin(list_users_cold_start[:int(split_cold_warm * len_new_user)]) == False)] = 0

/tmp/ipython-input-3859622317.py:4: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_train_groupby_user['n_interactions'][(df_train_groupby_user['user'].isin(list_users) == False) & (df_train_groupby_user['user'].isin(list_users_cold_start[:

In [ ]:
df_val = df_train.sort_values(by = ['timestamp']).groupby('user', as_index = False).agg(lambda x : [elem for elem in x][-2])

In [ ]:
df_train = df_train[df_train['user_item'].isin(df_val['user_item'].unique()) == False]

In [ ]:
df_test = df_train.sort_values(by = ['timestamp']).groupby('user', as_index = False).agg(lambda x : [elem for elem in x][-1])

In [ ]:
df_test = df_test[df_test['user'].isin(list_users)]

In [ ]:
df_train = df_train[df_train['user_item'].isin(df_test['user_item'].unique()) == False]

In [ ]:
def save_txt(data, filename):
    data_groupby_users = data.groupby("user", as_index = False).agg(lambda x : " ".join([str(elem) for elem in x]))
    data_groupby_users = data_groupby_users.sort_values(by = ["user"])
    final_data = data_groupby_users["user"].apply(lambda x : str(x) + "|") + data_groupby_users["item"]
    final_data = final_data.values
    final_data = "\n".join(final_data)
    f = open(filename, "w")
    f.write(final_data)
    f.close()
    print("Done")

In [ ]:
save_txt(df_train, path_save + "/train_for_test.txt")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/ColabNotebooks/ACIIDS2023GIFT4Rec/datasets/ml_1m_one_task/train_for_test.txt'

In [ ]:
save_txt(df_train, os.path.join(path_save, "train_for_test.txt"))

Done


In [ ]:
df_train.groupby('user', as_index = False).agg(lambda x : len(x))['item'].min()

22

In [ ]:
save_txt(df_test_cold, path_save + "/test_cold.txt")

Done


In [ ]:
save_txt(df_test_warm, path_save + "/test_warm.txt")

Done


In [ ]:
save_txt(df_train, path_save + "/train.txt")

Done


In [ ]:
save_txt(df_val, path_save + "/val.txt")

Done


In [ ]:
save_txt(df_test, path_save + "/test.txt")

Done


In [ ]:
if False:
    #df_train_copy = df_train.copy()
    #data_copy = data_copy[data_copy["task_id"] == 0]
    #save_kg_txt = df_train_copy["user"].apply(lambda x : str(x) + " ") + df_train_copy['rating'].astype(str) + df_train_copy["item"].apply(lambda x : " " + str(x))
    save_kg_txt = df_train_copy["user"].apply(lambda x : str(x) + " ") + "0" + df_train_copy["item"].apply(lambda x : " " + str(x))
    save_kg_txt = "\n".join(save_kg_txt.values)

In [ ]:
if True:
    #df_train_copy = df_train.copy()
    #data_copy = data_copy[data_copy["task_id"] == 0]
    #save_kg_txt = df_train_copy["user"].apply(lambda x : str(x) + " ") + df_train_copy['rating'].astype(str) + df_train_copy["item"].apply(lambda x : " " + str(x))
    #df_train_copy['rating'][df_train_copy['rating'] < 3] = 0
    #df_train_copy['rating'][df_train_copy['rating'] >= 3] = 1
    save_kg_txt = df_train_copy["user"].apply(lambda x : str(x) + " ") + df_train_copy['rating'].astype(int).astype(str) + df_train_copy["item"].apply(lambda x : " " + str(x))
    save_kg_txt = "\n".join(save_kg_txt.values)

In [ ]:
df_train_copy

,user_id,item_id,rating,timestamp,user,item,user_item
0,1,1193,5,978300760,3876,1193,3876|1193
1,1,661,3,978302109,3876,661,3876|661
2,1,914,3,978301968,3876,914,3876|914
3,1,3408,4,978300275,3876,3408,3876|3408
4,1,2355,5,978824291,3876,2355,3876|2355
...,...,...,...,...,...,...,...
1000203,6040,1090,3,956715518,842,1090,842|1090
1000205,6040,1094,5,956704887,842,1094,842|1094
1000206,6040,562,5,956704746,842,562,842|562
1000207,6040,1096,4,956715648,842,1096,842|1096


In [ ]:
save_kg_txt.split('\n')[0]

'3876 5 1193'

In [ ]:
df_train['user'].nunique()

5435

In [ ]:
f = open(path_save + "/kg_final.txt", "w")
f.write(save_kg_txt)
f.close()

In [ ]:
categorical_user_columns = ['categories']

In [ ]:
df_user['categories'] = df_user['gender'].astype(str) + ',' + df_user['occupation'].astype(str)

In [ ]:
df_user['user'] = df_user['user'].map(dict_user_to_id)

In [ ]:
df_train_groupby_user = df_train.groupby('user', as_index = False).agg(lambda x : len(x))

In [ ]:
#old
#df_train_groupby_user['n_interactions'] = df_train['item'].copy()
df_train_groupby_user['n_interactions'] = df_train_groupby_user['item'].copy()

In [ ]:
user_to_n_interactions = dict(df_train_groupby_user[['user', 'n_interactions']].values)

In [ ]:
df_user['n_interactions'] = df_user['user'].map(user_to_n_interactions)
df_user['n_interactions'][pd.isnull(df_user['n_interactions'])] = 0

/tmp/ipython-input-630573040.py:2: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_user['n_interactions'][pd.isnull(df_user['n_interactions'])] = 0
/tmp/ipython-input-630573040.py:2: SettingWithCopyWarning: 
A value is trying to be set on a

In [ ]:
numeric_user_columns = ['n_interactions', 'age']

In [ ]:
with open(path_save + '/user_numeric_name_columns.txt', 'w') as f:
  f.write('\t'.join(numeric_user_columns))
  f.close()

In [ ]:
with open(path_save + '/user_category_name_columns.txt', 'w') as f:
  f.write('\t'.join(categorical_user_columns))
  f.close()

In [ ]:
with open(path_save + '/item_numeric_name_columns.txt', 'w') as f:
  f.write('\t'.join([]))
  f.close()

In [ ]:
with open(path_save + '/item_category_name_columns.txt', 'w') as f:
  f.write('\t'.join([]))
  f.close()

In [ ]:
def preprocessing_categories_id(data, column, begin_index, all_category_keys = None):
    if all_category_keys is None:
        all_category_ids = data[column][pd.isnull(data[column]) == False].apply(lambda x : str(x) + ",").sum()
        all_category_ids = str(all_category_ids)[:-1]
        all_category_ids = list(set(all_category_ids.split(",")))
        try:
            all_category_ids = np.sort(np.asarray(all_category_ids).astype(int))
        except:
            all_category_ids = np.asarray(all_category_ids)
        all_category_ids = all_category_ids.astype(str)
        new_category_ids = np.linspace(0, all_category_ids.shape[0] - 1, all_category_ids.shape[0]).astype(int).reshape(-1, 1) + begin_index
        all_category_keys = dict(np.hstack((all_category_ids.reshape(-1, 1), new_category_ids)))
        new_begin_index = np.max(new_category_ids[:, 0]) + 1
    else:
        new_begin_index = max(list(all_category_keys.values())) + 1
    data[column][pd.isnull(data[column]) == False] = data[column][pd.isnull(data[column]) == False].apply(lambda x : "\t".join([str(all_category_keys[elem]) for elem in str(x).split(",")])  + "\t")

    data[column][pd.isnull(data[column])] = "0"

    return all_category_keys, data, new_begin_index

In [ ]:
begin_index = 0

In [ ]:
for column in categorical_user_columns:
    if True:
        category_keys, df_user, begin_index = preprocessing_categories_id(df_user, column, begin_index)

/tmp/ipython-input-4257381201.py:16: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  data[column][pd.isnull(data[column]) == False] = data[column][pd.isnull(data[column]) == False].apply(lambda x : "\t".join([str(all_category_keys[elem]) for e

In [ ]:
df_user["all_categories"] = df_user[categorical_user_columns].sum(axis = 1)
df_user["all_categories"] = df_user["all_categories"].apply(lambda x : x[:-1])

In [ ]:
df_user["all_categories"]

,all_categories
0,18\t11
1,1\t22
2,1\t17
3,1\t0
4,1\t21
...,...
6035,18\t17
6036,18\t5
6037,18\t5
6038,18\t2


In [ ]:
df_user.to_csv(path_save + '/user_features.csv', index = False)

In [ ]:
df_user = pd.read_csv('ml-1m/users.dat', sep = '::', names = ['user', 'gender', 'age', 'occupation', 'zipcode'], engine='python')

In [ ]:
df_user['user'] = df_user['user'].map(dict_user_to_id)

In [ ]:
!wget -O ml-1m/movies_extrainfos.dat "https://git.dml.ir/m.maheri/Melu_L2L_genericAlgo/raw/commit/00555ed067f71351ff74067971af022387b9baa6/movielens/ml-1m/movies_extrainfos.dat"

--2025-12-24 04:07:10--  https://git.dml.ir/m.maheri/Melu_L2L_genericAlgo/raw/commit/00555ed067f71351ff74067971af022387b9baa6/movielens/ml-1m/movies_extrainfos.dat
Resolving git.dml.ir (git.dml.ir)... 172.67.143.164, 104.21.39.70, 2606:4700:3032::ac43:8fa4, ...
Connecting to git.dml.ir (git.dml.ir)|172.67.143.164|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘ml-1m/movies_extrainfos.dat’

ml-1m/movies_extrai     [   <=>              ]   3.15M   731KB/s    in 7.4s    

2025-12-24 04:07:18 (435 KB/s) - ‘ml-1m/movies_extrainfos.dat’ saved [3305600]



In [ ]:
df_item = pd.read_csv('ml-1m/movies_extrainfos.dat', sep = '::', names = ['item', 'title', 'year', 'rate', 'released', 'genre', 'director', 'writer', 'actor', 'plot', 'poster'], engine='python', encoding="utf-8")

In [ ]:
def mapping(target_df, source_df, key):
    for column in source_df.columns:
        if key == column:
            continue
        target_df[column] = target_df[key].map(dict(source_df[[key, column]].values))
    return target_df

In [ ]:
df_train = mapping(df_train, df_user, 'user')
df_train = mapping(df_train, df_item, 'item')

In [ ]:
df_train

,user_id,item_id,rating,timestamp,user,item,user_item,gender,age,occupation,...,title,year,rate,released,genre,director,writer,actor,plot,poster
0,1,1193,5,978300760,3876,1193,3876|1193,F,1,10,...,One Flew Over the Cuckoo's Nest,1975.0,R,19 Nov 1975,Drama,Milos Forman,"Lawrence Hauben (screenplay), Bo Goldman (scre...","Michael Berryman, Peter Brocco, Dean R. Brooks...",McMurphy has a criminal past and has once agai...,https://m.media-amazon.com/images/M/MV5BZjA0OW...
1,1,661,3,978302109,3876,661,3876|661,F,1,10,...,James and the Giant Peach,1996.0,PG,12 Apr 1996,"Animation, Adventure, Family",Henry Selick,"Roald Dahl (based on the book by), Karey Kirkp...","Simon Callow, Richard Dreyfuss, Jane Leeves, J...",James' happy life at the English seaside is ru...,https://m.media-amazon.com/images/M/MV5BMTNkNW...
2,1,914,3,978301968,3876,914,3876|914,F,1,10,...,My Fair Lady,1964.0,G,25 Dec 1964,"Drama, Family, Musical",George Cukor,"Alan Jay Lerner (book), George Bernard Shaw (f...","Audrey Hepburn, Rex Harrison, Stanley Holloway...",Pompous phonetics professor Henry Higgins is s...,https://m.media-amazon.com/images/M/MV5BNGM0ZT...
3,1,3408,4,978300275,3876,3408,3876|3408,F,1,10,...,Erin Brockovich,2000.0,R,17 Mar 2000,"Biography, Drama",Steven Soderbergh,Susannah Grant,"Julia Roberts, David Brisbin, Dawn Didawick, A...",Erin Brockovich-Ellis is an unemployed single ...,https://m.media-amazon.com/images/M/MV5BYTA1NW...
4,1,2355,5,978824291,3876,2355,3876|2355,F,1,10,...,A Bug's Life,1998.0,G,25 Nov 1998,"Animation, Adventure, Comedy","John Lasseter, Andrew Stanton(co-director)","John Lasseter (original story by), Andrew Stan...","Dave Foley, Kevin Spacey, Julia Louis-Dreyfus,...","At an annual pace, a huge colony of ants is fo...",https://m.media-amazon.com/images/M/MV5BNThmZG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1000203,6040,1090,3,956715518,842,1090,842|1090,M,25,6,...,Platoon,1986.0,R,06 Feb 1987,"Drama, War",Oliver Stone,Oliver Stone,"Tom Berenger, Keith David, Willem Dafoe, Fores...","Chris Taylor is a young, naive American who gi...",https://m.media-amazon.com/images/M/MV5BM2Y1NT...
1000205,6040,1094,5,956704887,842,1094,842|1094,M,25,6,...,The Crying Game,1992.0,R,19 Feb 1993,"Crime, Drama, Romance",Neil Jordan,Neil Jordan,"Forest Whitaker, Miranda Richardson, Stephen R...",An unlikely kind of friendship develops betwee...,https://m.media-amazon.com/images/M/MV5BYWU4Mj...
1000206,6040,562,5,956704746,842,562,842|562,M,25,6,...,Welcome to the Dollhouse,1995.0,R,24 May 1996,"Comedy, Drama",Todd Solondz,Todd Solondz,"Heather Matarazzo, Victoria Davis, Christina B...",Seventh-grade is no fun. Especially for Dawn W...,https://ia.media-imdb.com/images/M/MV5BOGE0N2U...
1000207,6040,1096,4,956715648,842,1096,842|1096,M,25,6,...,Sophie's Choice,1982.0,R,04 Mar 1983,"Drama, Romance",Alan J. Pakula,"William Styron (based on the novel: ""Sophie's ...","Meryl Streep, Kevin Kline, Peter MacNicol, Rit...",Sophie is the survivor of Nazi concentration c...,https://m.media-amazon.com/images/M/MV5BMTk3ND...


In [ ]:
df_test_cold

,user_id,item_id,rating,timestamp,user,item,user_item
233,4,3468,5,978294008,5756,3468,5756|3468
234,4,1210,3,978293924,5756,1210,5756|1210
235,4,2951,4,978294282,5756,2951,5756|2951
236,4,1214,4,978294260,5756,1214,5756|1214
237,4,1036,4,978294282,5756,1036,5756|1036
...,...,...,...,...,...,...,...
999738,6038,1296,5,956714684,5892,1296,5892|1296
999739,6038,1354,3,956714725,5892,1354,5892|1354
999742,6038,2716,3,956707604,5892,2716,5892|2716
999743,6038,3396,3,956706827,5892,3396,5892|3396


In [ ]:
df_test = pd.concat([df_test, df_test_warm])
df_test = mapping(df_test, df_user, 'user')
df_test = mapping(df_test, df_item, 'item')

In [ ]:
df_test_cold

,user_id,item_id,rating,timestamp,user,item,user_item
233,4,3468,5,978294008,5756,3468,5756|3468
234,4,1210,3,978293924,5756,1210,5756|1210
235,4,2951,4,978294282,5756,2951,5756|2951
236,4,1214,4,978294260,5756,1214,5756|1214
237,4,1036,4,978294282,5756,1036,5756|1036
...,...,...,...,...,...,...,...
999738,6038,1296,5,956714684,5892,1296,5892|1296
999739,6038,1354,3,956714725,5892,1354,5892|1354
999742,6038,2716,3,956707604,5892,2716,5892|2716
999743,6038,3396,3,956706827,5892,3396,5892|3396


In [ ]:
df_test_cold = mapping(df_test_cold, df_user, 'user')
df_test_cold = mapping(df_test_cold, df_item, 'item')

/tmp/ipython-input-1499374598.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_df[column] = target_df[key].map(dict(source_df[[key, column]].values))
/tmp/ipython-input-1499374598.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  target_df[column] = target_df[key].map(dict(source_df[[key, column]].values))
/tmp/ipython-input-1499374598.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the

In [ ]:
df_test_cold['zipcode']

,zipcode
233,02460
234,02460
235,02460
236,02460
237,02460
...,...
999738,14706
999739,14706
999742,14706
999743,14706


In [ ]:
#df_test_cold['rate'] = df_test_cold['rating'].copy()
#df_test['rate'] = df_test['rating'].copy()
#df_train['rate'] = df_train['rating'].copy()
df_test_cold.to_csv(path_save + '/test_cold.csv', index = False)
df_test.to_csv(path_save + '/test.csv', index = False)
df_train.to_csv(path_save + '/train.csv', index = False)

In [ ]:
df_user.to_csv(path_save + '/user.csv', index = False)
df_item.to_csv(path_save + '/item.csv', index = False)

In [ ]:
#!python main_kgat_2.py --data_name ml_1m_one_task --use_pretrain 0 --cf_print_every 16 --kg_print_every 16 --evaluate_every 1 --Ks '[20, 50]' --cf_batch_size 1024  --lr 1e-4 --masking_train 1 --use_weight_embeds 1 --training_embed_weight 1 --use_train_data_to_train_weight_embed 1 --use_weight_embeds_no_grad 1

In [ ]:
#!python main_kgat_2.py --data_name ml_1m_one_task --use_pretrain 0 --cf_print_every 16 --kg_print_every 16 --evaluate_every 1 --Ks '[20, 50]' --cf_batch_size 1024  --lr 1e-3 --masking_train 1 --use_weight_embeds 1 --training_embed_weight 1 --use_train_data_to_train_weight_embed 1 --full_sampling 1

In [ ]:
#!python main_kgat_2.py --data_name ml_1m_one_task --use_pretrain 0 --cf_print_every 16 --kg_print_every 16 --evaluate_every 1 --Ks '[20, 50]' --cf_batch_size 1024  --lr 1e-2 --masking_train 1 --use_weight_embeds 1 --training_embed_weight 1 --use_train_data_to_train_weight_embed 0 --full_sampling 1

In [ ]:
#!python main_kgat_2.py --data_name ml_1m_one_task --use_pretrain 0 --cf_print_every 16 --kg_print_every 16 --evaluate_every 1 --Ks '[20, 50]' --cf_batch_size 1024  --lr 1e-2 --masking_train 1 --use_weight_embeds 1 --training_embed_weight 0 --use_train_data_to_train_weight_embed 0 --full_sampling 1

In [ ]:
#!python main_kgat_2.py --data_name ml_1m_one_task --use_pretrain 0 --cf_print_every 16 --kg_print_every 16 --evaluate_every 1 --Ks '[20, 50]' --cf_batch_size 1024  --lr 1e-2 --masking_train 1 --use_weight_embeds 0 --training_embed_weight 0 --use_train_data_to_train_weight_embed 0 --full_sampling 1

In [ ]:
#!python main_kgat_raw.py --data_name ml_1m_one_task --use_pretrain 0 --cf_print_every 16 --kg_print_every 16 --evaluate_every 1 --Ks '[20, 50]' --cf_batch_size 1024  --lr 1e-2 --masking_train 1

In [ ]:
#!python main_kgat_2_new_backward_2.py --data_name ml_1m_one_task --use_pretrain 0 --cf_print_every 16 --kg_print_every 16 --evaluate_every 1 --Ks '[20,50]' --cf_batch_size 1024  --lr 1e-3 --masking_train 1 --use_weight_embeds 1 --training_embed_weight 1 --use_train_data_to_train_weight_embed 0 --stopping_steps 20 --train_weight_embed_lr 1e-3 --use_binary_weight 0 --training_embed_weight_batch_size 200 --stop_training_embed_weight_epoch 1000